# Hierarchical Clustering

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
# Load the dataset
X = pd.read_csv('finalclusteringdataset.csv')
print(X.head())
print(X.shape)

In [ ]:
# Calculate linkage matrix (using Ward's method)
linkage_matrix = linkage(X, method='ward')
print('Linkage matrix computed using Ward method')

In [ ]:
# Plot dendrogram
plt.figure(figsize=(15, 7))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)  # Show only last 30 merges
plt.title('Hierarchical Clustering Dendrogram (Truncated)')
plt.xlabel('Sample Index or (Cluster Size)')
plt.ylabel('Distance')
plt.axhline(y=50, c='red', linestyle='--', label='Cut threshold')  # Example cut
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Extract clusters using different numbers
# Test multiple k values to find optimal
best_score = -1
best_k = 2
scores = {}

for k in range(2, 11):
    cluster_labels = fcluster(linkage_matrix, k, criterion='maxclust')
    score = silhouette_score(X, cluster_labels)
    scores[k] = score
    if score > best_score:
        best_score = score
        best_k = k

print(f'Best k: {best_k} with Silhouette score: {best_score:.4f}')
print('\nSilhouette scores for different k:')
for k, score in scores.items():
    print(f'k={k}: {score:.4f}')

In [ ]:
# Apply clustering with optimal k
cluster_labels = fcluster(linkage_matrix, best_k, criterion='maxclust')

print(f'Number of clusters: {best_k}')
print(f'Silhouette Score: {silhouette_score(X, cluster_labels):.4f}')
print(f'Davies-Bouldin Score: {davies_bouldin_score(X, cluster_labels):.4f}')

In [ ]:
# Visualize clusters (2D projection)
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=50, alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Hierarchical Clustering Results')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Test different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
silhouette_scores_by_method = {}

plt.figure(figsize=(14, 10))
for i, method in enumerate(linkage_methods):
    linkage_mat = linkage(X, method=method)
    cluster_labels_method = fcluster(linkage_mat, best_k, criterion='maxclust')
    score = silhouette_score(X, cluster_labels_method)
    silhouette_scores_by_method[method] = score
    
    plt.subplot(2, 2, i+1)
    X_pca_temp = pca.fit_transform(X)
    plt.scatter(X_pca_temp[:, 0], X_pca_temp[:, 1], c=cluster_labels_method, cmap='viridis', s=50, alpha=0.6)
    plt.title(f'{method.capitalize()} Linkage (Score: {score:.4f})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nSilhouette scores by linkage method:')
for method, score in silhouette_scores_by_method.items():
    print(f'{method}: {score:.4f}')

In [ ]:
# Cluster distribution
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(unique, counts, color='skyblue', edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Number of Data Points')
plt.title('Hierarchical Clustering Distribution')
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)
plt.show()

print('\nCluster Distribution:')
for cluster, count in zip(unique, counts):
    print(f'Cluster {cluster}: {count} samples ({count/len(X)*100:.2f}%)')

In [ ]:
# Save results
X['Cluster'] = cluster_labels
X.to_csv('hierarchical_results.csv', index=False)
print('Hierarchical clustering results saved to hierarchical_results.csv')